# Module 07 — Walk-Forward Out-of-Sample Backtest

Final 30% out-of-sample walk-forward backtest. Formation parameters and the 40 structurally eligible pairs are frozen from the training sample.


In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from src.backtest import run_walk_forward_backtest, backtest_summary

pd.set_option("display.max_columns", 100)


## 1. Load frozen formation data


In [2]:
train_prices = pd.read_parquet("data/processed/train_prices.parquet")
test_prices = pd.read_parquet("data/processed/test_prices.parquet")
eligible_pairs = pd.read_parquet("data/processed/eligible_pairs.parquet")
cointegrated_pairs = pd.read_parquet("data/processed/cointegrated_pairs.parquet")

assert isinstance(train_prices.index, pd.DatetimeIndex), "train_prices index is not DatetimeIndex."
assert isinstance(test_prices.index, pd.DatetimeIndex), "test_prices index is not DatetimeIndex."
assert train_prices.index.max() < test_prices.index.min(), "Train/test periods overlap."

print("Train:", train_prices.index.min(), "->", train_prices.index.max(), train_prices.shape)
print("Test :", test_prices.index.min(), "->", test_prices.index.max(), test_prices.shape)
print("Eligible pairs:", len(eligible_pairs))


Train: 2016-01-04 00:00:00 -> 2022-12-27 00:00:00 (1759, 466)
Test : 2022-12-28 00:00:00 -> 2025-12-31 00:00:00 (755, 466)
Eligible pairs: 40


## 2. Historical risk-free rate


In [3]:
rf_data = pd.read_parquet("data/processed/risk_free_rates.parquet")

if isinstance(rf_data, pd.DataFrame):
    if rf_data.shape[1] != 1:
        raise ValueError("risk_free_rates.parquet must contain exactly one rate column.")
    risk_free_rates = rf_data.iloc[:, 0]
else:
    risk_free_rates = pd.Series(rf_data)

risk_free_rates.index = pd.to_datetime(risk_free_rates.index)
risk_free_rates = risk_free_rates.sort_index().astype(float)

if risk_free_rates.abs().median() > 1:
    raise ValueError("Risk-free rates appear to be percentages. Divide them by 100 first.")

risk_free_rates.tail()


Date
2025-12-24    0.03555
2025-12-26    0.03543
2025-12-29    0.03538
2025-12-30    0.03540
2025-12-31    0.03547
Name: risk_free_rate, dtype: float64

## 3. Final backtest configuration


In [4]:
INITIAL_CAPITAL = 100_000.0
ENTRY_Z = 1.5
TARGET_PROBABILITY = 0.70
MEMORY_WINDOW = 60
MAX_HORIZON_DAYS = 126
N_PATHS = 5000
EWMA_LAMBDA = 0.94
SEED = 42


## 4. Run the walk-forward simulation


In [5]:
results = run_walk_forward_backtest(
    train_prices=train_prices,
    test_prices=test_prices,
    eligible_pairs=eligible_pairs,
    cointegrated_pairs=cointegrated_pairs,
    risk_free_rates=risk_free_rates,
    initial_capital=INITIAL_CAPITAL,
    entry_z=ENTRY_Z,
    target_probability=TARGET_PROBABILITY,
    memory_window=MEMORY_WINDOW,
    max_horizon_days=MAX_HORIZON_DAYS,
    n_paths=N_PATHS,
    ewma_lambda=EWMA_LAMBDA,
    seed=SEED,
)

trades = results["trades"]
equity_curve = results["equity_curve"]
skipped_signals = results["skipped_signals"]

print("Completed trades:", len(trades))
print("Skipped records :", len(skipped_signals))
print("Final equity    :", equity_curve["equity"].iloc[-1])


Completed trades: 254
Skipped records : 585
Final equity    : 132661.06455480028


## 5. Results and sanity checks


In [6]:
summary = backtest_summary(
    trades=trades,
    equity_curve=equity_curve,
    initial_capital=INITIAL_CAPITAL,
)
summary


initial_capital             100000.000000
final_equity                132661.064555
total_return                     0.326611
max_drawdown                    -0.510505
n_trades                       254.000000
win_rate                         0.488189
average_trade_return             0.077734
median_trade_return             -0.040611
max_concurrent_positions        34.000000
Name: module_07_summary, dtype: float64

In [7]:
if not trades.empty:
    display(
        trades[[
            "pair", "entry_date", "exit_date", "entry_z",
            "convergence_horizon_trading_days", "option_calendar_dte",
            "dependent_contracts", "independent_contracts",
            "entry_premium", "exit_value", "pnl",
            "trade_return", "exit_reason",
        ]].head(20)
    )

if not skipped_signals.empty and "reason" in skipped_signals.columns:
    display(skipped_signals["reason"].value_counts())


,pair,entry_date,exit_date,entry_z,convergence_horizon_trading_days,option_calendar_dte,dependent_contracts,independent_contracts,entry_premium,exit_value,pnl,trade_return,exit_reason
0,PNR-NWSA,2022-12-28,2023-02-01,-1.616817,97,141,1,3,724.913654,1534.998812,810.085158,1.117492,convergence
1,DOV-CDW,2023-01-19,2023-02-01,-1.743506,87,127,1,1,1846.838094,3213.374390,1366.536296,0.739933,convergence
2,EMR-TEL,2022-12-28,2023-02-08,3.194643,121,176,2,1,1854.502076,4020.321904,2165.819829,1.167871,convergence
3,IFF-SWK,2023-01-10,2023-02-09,1.615938,79,115,3,1,2240.149665,7096.561529,4856.411864,2.167896,convergence
4,MCO-ZTS,2023-01-18,2023-02-15,1.609153,70,103,1,1,3272.846029,3742.557368,469.711339,0.143518,convergence
5,MLM-VMC,2023-02-09,2023-03-09,-1.588086,45,67,1,3,3616.299554,5078.726281,1462.426728,0.404399,convergence
6,SHW-HD,2022-12-28,2023-03-27,-2.193745,60,89,1,1,2765.070223,2962.054443,196.984221,0.071240,expiry
7,URI-MS,2023-01-06,2023-03-27,1.536003,54,80,1,4,3355.721726,317.321777,-3038.399949,-0.905439,expiry
8,WMT-SPGI,2022-12-28,2023-04-20,1.514515,78,113,14,1,4443.722605,2225.897217,-2217.825388,-0.499092,expiry
9,SHW-DHI,2022-12-28,2023-04-26,-1.862157,82,119,1,3,3331.269139,0.000000,-3331.269139,-1.000000,expiry


reason
insufficient_cash    585
Name: count, dtype: int64

In [8]:
if not trades.empty:
    print("Largest single entry premium:", trades["entry_premium"].max())
    print("Median entry premium:", trades["entry_premium"].median())

print("Maximum concurrent positions:", equity_curve["n_open_positions"].max())
print("Minimum cash balance:", equity_curve["cash"].min())


Largest single entry premium: 24707.43242790731
Median entry premium: 3273.366725467636
Maximum concurrent positions: 34
Minimum cash balance: 56.755108592416946


## 6. Save Module 07 outputs


In [9]:
trades.to_parquet("data/processed/trades.parquet", index=False)
equity_curve.to_parquet("data/processed/equity_curve.parquet", index=True)
skipped_signals.to_parquet("data/processed/skipped_signals.parquet", index=False)

print("Saved Module 07 outputs.")


Saved Module 07 outputs.
